# TP DES & AES — Partie 1 & 2 : chiffrement de Feistel personnalisé


## 1. Conception

On conçoit un chiffrement par blocs basé sur un **réseau de Feistel**. Le principe : on coupe le bloc en deux moitiés $L$ et $R$, et à chaque tour :

$$L_{i+1} = R_i \qquad R_{i+1} = L_i \oplus F(R_i, K_i)$$

L'intérêt de cette structure est que le déchiffrement est le même algorithme avec les sous-clés dans l'ordre inverse, **même si $F$ n'est pas inversible**.

### Choix des paramètres

| Paramètre | Choix | Justification |
|---|---|---|
| Taille de bloc | 64 bits (2 × 32 bits) | Même taille que DES pour une comparaison directe, avec deux moitiés faciles à traiter en Python |
| Taille de clé | 128 bits | 56 bits (DES) est aujourd’hui insuffisant face à la force brute ; la clé proposée fait 128 bits, ce qui ne garantit pas la sécurité de ce chiffrement maison |
| Nombre de tours | 16 | 16 tours, comme DES, permettent d’observer la diffusion sur plusieurs tours ; les mesures d’avalanche ci-dessous sont empiriques et ne constituent pas une preuve de sécurité |
| Mode | Cipher Block Chaining (CBC) + padding PKCS#7 | Un vecteur d'initialisation IV aléatoire et renouvelé à chaque chiffrement, puis le chaînage avec le bloc chiffré précédent, empêchent que des blocs identiques produisent les mêmes blocs chiffrés |

### Fonction de tour $F(R, K)$

#### Vue d'ensemble

À chaque tour, la fonction de tour transforme $R$ et la sous-clé $K$ selon le chemin suivant :

$$R,K \xrightarrow{\text{addition}} x \xrightarrow{\text{confusion}} y \xrightarrow{\text{diffusion}} F(R,K)$$

**Rôle des trois étapes** :

- L'**addition** mélange les valeurs de $R$ et $K$ avec des retenues.
- La **S-box** mélange les 4-bit blocs de façon non linéaire, d'où la **confusion**.
- La **couche linéaire** redistribue les bits sur tout le mot de 32 bits, d'où la **diffusion**.

Ces propriétés permettent à une modification locale de produire progressivement une modification globale du bloc sur plusieurs tours.

#### 1. Addition modulaire

$$x=(R+K)\bmod 2^{32}$$

Contrairement au XOR, une addition peut produire des **retenues**. Cette propagation rend la relation entre les entrées, la clé et $x$ plus difficile à lire comme une combinaison indépendante de chaque bit. L'addition est donc **non linéaire par rapport au XOR**.

Le résultat modulo $2^{32}$ permet de maintenir la représentation sur exactement 32 bits.

#### 2. Confusion par la S-box

Le mot $x$ est découpé en **8 nibbles** de 4 bits : $32\text{ bits}=8\times4\text{ bits}$

Chaque nibble passe ensuite dans une **S-box**, une table qui remplace une entrée de 4 bits par une sortie de 4 bits :

$$4\text{ bits}\longrightarrow4\text{ bits}$$

Le code utilise la table du chiffrement léger **PRESENT**, conçu pour les appareils à faible capacité, avec des blocs de 64 bits. Ici, on emprunte seulement sa S-box 4×4 et non l'intégralité du chiffrement. L'indice de la table est l'entrée et la valeur est la sortie.

#### 3. Diffusion par rotations

$$z=y\oplus\operatorname{rotl}(y,5)\oplus\operatorname{rotl}(y,11)$$

`rotl(y, s)` fait tourner les 32 bits de $y$ de `s` positions vers la gauche. Les bits qui dépassent à gauche reviennent à droite.

Cette redistribution est la **diffusion** : une modification dans un nibble de $y$ influence plusieurs zones du mot de sortie $z$. Au tour suivant, cette sortie est mélangée avec $L$, puis à nouveau transformée par une autre sous-clé. Sur plusieurs tours, une petite modification peut se propager à presque tout le bloc (effet d'avalanche).

### Cadencement de clé

La clé de 128 bits est découpée en 4 mots $W_0..W_3$ de 32 bits. On garde un état $k$ :

$$k_{i} = \text{rotl}\big((k_{i-1} + W_{i \bmod 4} + C\cdot(i+1)) \bmod 2^{32},\ 7\big)$$

avec $C = \texttt{0x9E3779B9}$, constante du nombre d'or utilisée dans TEA (Tiny Encryption Algorithm, un algorithme de chiffrement symétrique par blocs). La constante dépendant du tour et l’addition suivie d’une rotation font varier les sous-clés d’un tour à l’autre. Ce cadencement simple n’a pas fait l’objet d’une analyse cryptographique.

In [13]:
import os
import random

M32 = 0xFFFFFFFF
SBOX = [0xC, 0x5, 0x6, 0xB, 0x9, 0x0, 0xA, 0xD, 0x3, 0xE, 0xF, 0x8, 0x4, 0x7, 0x1, 0x2]  # S-box PRESENT
ROUNDS = 16
C = 0x9E3779B9

def rotl(x, s):
    return ((x << s) | (x >> (32 - s))) & M32

def F(R, K):
    x = (R + K) & M32                     # addition modulaire
    y = 0
    for i in range(8):                    # S-box sur chaque nibble
        y |= SBOX[(x >> (4 * i)) & 0xF] << (4 * i)
    return y ^ rotl(y, 5) ^ rotl(y, 11)   # diffusion

def key_schedule(key, rounds=ROUNDS):
    """Une clé de 16 octets donne une liste de sous-clés de 32 bits."""
    if not isinstance(key, (bytes, bytearray)) or len(key) != 16:
        raise ValueError("La clé doit contenir exactement 16 octets")
    if not isinstance(rounds, int) or rounds < 1:
        raise ValueError("Le nombre de tours doit être un entier positif")
    W = [int.from_bytes(key[4*i:4*i+4], 'big') for i in range(4)]
    k, sub = 0, []
    for i in range(rounds):
        k = rotl((k + W[i % 4] + C * (i + 1)) & M32, 7)
        sub.append(k)
    return sub

### Chiffrement / déchiffrement d'un bloc

In [14]:
def feistel_block(block, subkeys):
    L, R = block >> 32, block & M32
    for K in subkeys:
        L, R = R, L ^ F(R, K)
    return (R << 32) | L        # on échange les moitiés à la fin

def encrypt_block(block, subkeys):
    return feistel_block(block, subkeys)

def decrypt_block(block, subkeys):
    return feistel_block(block, subkeys[::-1])   # mêmes opérations, clés inversées

### Messages de taille quelconque : padding PKCS#7 + CBC

On complète le message avec $n$ octets valant $n$ ($1 \le n \le 8$) pour obtenir un multiple de 8 octets. Un message vide donne donc un bloc complet de padding. Le mode CBC utilise un IV aléatoire de 8 octets, préfixé au chiffré, puis chaque bloc clair est combiné par XOR avec le bloc chiffré précédent.
**Limite :** CBC avec padding n’authentifie pas le message. Une altération du chiffré peut modifier le clair et les erreurs de padding peuvent révéler de l’information. Cette construction sert à illustrer Feistel, le padding et CBC ; pour un usage réel, employer une bibliothèque reconnue avec un mode authentifié (par exemple AES-GCM).

In [15]:
def pad(data):
    if not isinstance(data, (bytes, bytearray)):
        raise TypeError("Le message doit être de type bytes ou bytearray")
    n = 8 - len(data) % 8
    return bytes(data) + bytes([n]) * n

def unpad(data):
    if not data or len(data) % 8 != 0:
        raise ValueError("Les données paddées doivent contenir des blocs complets")
    n = data[-1]
    if not 1 <= n <= 8 or data[-n:] != bytes([n]) * n:
        raise ValueError("Padding PKCS#7 invalide")
    return data[:-n]

def encrypt(data, key):
    sk = key_schedule(key)
    data = pad(data)
    iv = os.urandom(8)
    previous = iv
    out = bytearray()
    for i in range(0, len(data), 8):
        block = bytes(a ^ b for a, b in zip(data[i:i+8], previous))
        b = int.from_bytes(block, 'big')
        encrypted = encrypt_block(b, sk).to_bytes(8, 'big')
        out += encrypted
        previous = encrypted
    return iv + bytes(out)

def decrypt(data, key):
    if not isinstance(data, (bytes, bytearray)):
        raise TypeError("Le chiffré doit être de type bytes ou bytearray")
    if len(data) < 16 or len(data) % 8 != 0:
        raise ValueError("Un chiffré CBC doit contenir un IV et au moins un bloc complet")
    sk = key_schedule(key)
    iv, ciphertext = data[:8], data[8:]
    previous = iv
    out = bytearray()
    for i in range(0, len(ciphertext), 8):
        current = ciphertext[i:i+8]
        b = int.from_bytes(current, 'big')
        decrypted = decrypt_block(b, sk).to_bytes(8, 'big')
        out += bytes(a ^ b for a, b in zip(decrypted, previous))
        previous = current
    return unpad(bytes(out))

In [16]:
key = os.urandom(16)
msg = b"Bonjour INSA Lyon !"
ct = encrypt(msg, key)
print("clé       :", key.hex())
print("clair     :", msg)
print("chiffré   :", ct.hex())
print("déchiffré :", decrypt(ct, key))

clé       : 679f86f7003c72c4a6e345dad770528a
clair     : b'Bonjour INSA Lyon !'
chiffré   : f39605cb25222cbefe1d00ae816d9a5fcdbfb24e51cad10059e005ac45ea49fc
déchiffré : b'Bonjour INSA Lyon !'


### Diffusion et confusion : effet avalanche

Pour vérifier la diffusion, on change **1 bit** du clair (ou de la clé) et on compte le nombre de bits du chiffré qui changent. Un chiffrement avec une bonne diffusion tend à en changer environ la moitié, soit **32 bits sur 64**. Cette mesure isolée ne suffit pas à évaluer la sécurité.

On compare avec un chiffrement trivial « XOR avec la clé » pour montrer la différence.

In [17]:
def hamming(a, b):
    return bin(a ^ b).count('1')

def avalanche(rounds, n=500):
    tot_pt, tot_key = 0, 0
    for _ in range(n):
        key = os.urandom(16)
        sk = key_schedule(key, rounds)
        p = random.getrandbits(64)
        bit = 1 << random.randrange(64)
        tot_pt += hamming(feistel_block(p, sk), feistel_block(p ^ bit, sk))
        # 1 bit de clé modifié
        k2 = bytearray(key)
        k2[random.randrange(16)] ^= 1 << random.randrange(8)
        tot_key += hamming(feistel_block(p, sk), feistel_block(p, key_schedule(bytes(k2), rounds)))
    return tot_pt / n, tot_key / n

print("tours | bits changés (1 bit clair) | bits changés (1 bit clé)")
for r in [1, 2, 3, 4, 6, 8, 16]:
    a, b = avalanche(r)
    print(f"{r:5d} | {a:26.1f} | {b:24.1f}")

# chiffrement trivial XOR
k = random.getrandbits(64)
p = random.getrandbits(64)
print("\nXOR trivial : bits changés pour 1 bit de clair =", hamming(p ^ k, (p ^ 1) ^ k))

tours | bits changés (1 bit clair) | bits changés (1 bit clé)
    1 |                        5.1 |                      2.0
    2 |                       15.9 |                      8.8
    3 |                       26.1 |                     15.5
    4 |                       31.2 |                     22.8
    6 |                       31.9 |                     31.8
    8 |                       31.9 |                     32.3
   16 |                       32.4 |                     32.1

XOR trivial : bits changés pour 1 bit de clair = 1


**Interprétation :** à 1 tour, seule une moitié est recopiée telle quelle ; les tours suivants propagent progressivement la différence. Dans cet échantillon, la moyenne se rapproche de 32 bits à partir de 6 tours. Le résultat dépend des clés et des entrées testées : l’effet avalanche est une propriété utile, mais il ne prouve pas qu’un algorithme est sûr.

Le XOR trivial ne change qu’un bit : il n’apporte aucune diffusion. Le chiffrement Feistel proposé illustre une construction plus élaborée, mais reste un exercice pédagogique non expertisé et ne doit pas être utilisé pour protéger des données réelles.

## 2. Vérification de la correction

On génère des clés aléatoires et on teste des messages de longueurs et de contenus variés, y compris des cas limites. Pour chaque message on vérifie que `decrypt(encrypt(m)) == m` avec 10 clés aléatoires par message.

In [18]:
tests = {
    "vide":                 b"",
    "1 octet":              b"A",
    "7 octets (impair)":    b"abcdefg",
    "8 octets (1 bloc)":    b"12345678",
    "9 octets":             b"123456789",
    "texte ASCII":          b"The quick brown fox jumps over the lazy dog",
    "UTF-8 (accents)":      "Chiffrement à clé symétrique, été, ça marche ?".encode(),
    "motif répété":         b"ABCDEFGH" * 8,
    "que des zéros":        bytes(32),
    "que des 0xFF":         b"\xff" * 33,
    "tous les octets":      bytes(range(256)),
    "binaire aléatoire":    os.urandom(1001),
    "gros message (100 ko)": os.urandom(100_000),
}

print(f"{'test':24s} {'taille':>7s} {'chiffré':>8s}  résultat")
for name, m in tests.items():
    ok = all(decrypt(encrypt(m, k), k) == m for k in [os.urandom(16) for _ in range(10)])
    c = encrypt(m, os.urandom(16))
    print(f"{name:24s} {len(m):7d} {len(c):8d}  {'OK' if ok else 'ERREUR'}")

test                      taille  chiffré  résultat
vide                           0       16  OK
1 octet                        1       16  OK
7 octets (impair)              7       16  OK
8 octets (1 bloc)              8       24  OK
9 octets                       9       24  OK
texte ASCII                   43       56  OK
UTF-8 (accents)               52       64  OK
motif répété                  64       80  OK
que des zéros                 32       48  OK
que des 0xFF                  33       48  OK
tous les octets              256      272  OK
binaire aléatoire           1001     1016  OK
gros message (100 ko)     100000   100016  OK


Les cas testés ci-dessus retrouvent exactement le message original, y compris le message vide (qui donne un bloc de padding) et les longueurs non multiples de 8. Ces essais vérifient la correction fonctionnelle pour les échantillons choisis, pas la sécurité cryptographique.